# Sesión 2 · El resto de primitivas

**Curso MCP · servidores remotos** — notebook 2 de 4

En la sesión anterior viste **tools**: acciones que el modelo invoca. Ahora las otras tres
formas que tiene un servidor de aportar algo, y cada una responde a una pregunta distinta.

| Primitiva | ¿Quién la dispara? | ¿Para qué? | Minutos |
|---|---|---|:--:|
| **Resources** | El host | Datos que se traen como contexto | 30 |
| **Prompts** | El usuario | Workflows enlatados | 20 |
| **Interacción** | El servidor, a mitad de un tool | Pedir algo que falta | 25 |
| **Progreso** | El servidor, durante la llamada | No dejar a nadie a ciegas | 20 |

**Preparación de la sesión.** Instala el SDK e importa `Client`.

**Edita `MI_URL`** con la dirección de tu servidor, la que anotaste al final de la sesión 1.
Todas las celdas de este notebook la usan.

In [ ]:
!pip install --quiet "mcp==2.0.0"

# La URL de tu servidor, desplegado en la sesión 1.
MI_URL = "https://curso-mcp-XXXXX.europe-west1.run.app/mcp"  # ← EDITAR

from mcp import Client

## 1. Resources

Un **tool** lo llama el modelo cuando decide que lo necesita. Un **resource** lo trae el
**host** y lo pone en el contexto, normalmente porque el usuario lo ha elegido.

La diferencia no es técnica, es de control: los resources se parecen a adjuntar un fichero a
la conversación; los tools, a darle un botón al modelo.

**Regla práctica para decidir:** si tiene efectos o toma decisiones, es un tool. Si es material
de lectura, es un resource.

**Los resources que publica el servidor**, en sus dos formas. `list_resources()` devuelve los
que existen como tales; `list_resource_templates()`, los parametrizados.

**Vas a ver** un resource fijo (`catalogo://tablas`), uno de interfaz (`ui://`, que sale en la
sesión 4) y una plantilla con un hueco `{tabla}`.

In [ ]:
async def ver_resources():
    async with Client(MI_URL) as c:
        fijos = await c.list_resources()
        print("Resources fijos:")
        for r in fijos.resources:
            print(f"  · {r.uri} — {r.name}")

        plantillas = await c.list_resource_templates()
        print("\nPlantillas de URI:")
        for t in plantillas.resource_templates:
            print(f"  · {t.uri_template}")

await ver_resources()

Las **plantillas** son resources parametrizados. `catalogo://tablas/{tabla}` no es un recurso
concreto sino una familia: el host rellena `{tabla}` y obtiene el que quiera, sin que el
servidor tenga que enumerarlos todos por adelantado.

**Leer un resource.** `read_resource()` recibe el URI y devuelve su contenido; como este lo
declaramos de tipo `text/markdown`, sale texto.

**Vas a ver** el catálogo de tablas, generado consultando BigQuery en el momento de la lectura.

In [ ]:
async def leer(uri):
    async with Client(MI_URL) as c:
        r = await c.read_resource(uri)
        return r.contents[0].text

print(await leer("catalogo://tablas"))

**Rellenar la plantilla.** Aquí se ve para qué sirven: `catalogo://tablas/{tabla}` no era un
recurso sino una familia. Sustituyes el hueco y obtienes el que quieras, sin que el servidor
tenga que enumerar por adelantado uno por tabla.

**Vas a ver** el esquema de `bikeshare_trips` en una tabla de Markdown.

In [ ]:
# Y ahora una instancia de la plantilla
print(await leer("catalogo://tablas/bikeshare_trips"))

### `ttlMs` y `cacheScope`: cacheo como decisión de diseño

La revisión `2026-07-28` añade dos campos a todo lo que se lista o se lee:

- **`ttlMs`** — cuántos milisegundos puede el cliente dar por buena esta respuesta.
- **`cacheScope`** — `"public"` si un intermediario compartido puede cachearla, `"private"`
  si es específica de este usuario.

En un curso genérico esto sería una nota al pie. **Aquí no**, y conviene entender por qué:
cada lectura de `catalogo://tablas` es una llamada a la API de BigQuery. Sin `ttlMs`, un host
que refresque su panel cada pocos segundos te genera tráfico y coste por un catálogo que
cambia una vez al mes.

En un servidor MCP remoto **quien paga las consultas es el dueño del servidor**, no quien las
pide. El cacheo deja de ser una optimización y pasa a ser control de gasto.

**Las pistas de cacheo que manda el servidor.** Viajan en la propia respuesta de
`list_resources`, al lado de los datos.

**Vas a ver** dos valores: cuántos milisegundos puede el cliente reutilizar esta respuesta, y
si un intermediario compartido tiene permiso para guardarla.

In [ ]:
async def ver_cache():
    async with Client(MI_URL) as c:
        r = await c.list_resources()
        print("ttlMs:", r.ttl_ms, "· cacheScope:", r.cache_scope)

await ver_cache()

## 2. Prompts

Un **prompt** es una plantilla parametrizada que dispara **el usuario**, no el modelo.
Normalmente aparecen en el host como comandos: `/explorar`.

Sirven para encapsular *la forma correcta de preguntar* a tus datos. Tú sabes qué tablas hay
que cruzar y en qué orden; el usuario no tiene por qué.

**Los prompts, listados y desplegados.** Primero cuáles hay y con qué argumentos; después
`get_prompt()` rellena uno.

Fíjate en lo que devuelve: **no una respuesta, sino el mensaje** que se le va a dar al modelo.
Un prompt es andamiaje de conversación, no lógica de negocio.

**Vas a ver** el prompt `explorar` y el texto que genera para la tabla que le pasamos.

In [ ]:
async def ver_prompts():
    async with Client(MI_URL) as c:
        lista = await c.list_prompts()
        for p in lista.prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"· {p.name}({args}) — {p.description}")

        detalle = await c.get_prompt("explorar", {"tabla": "bikeshare_trips"})
        print("\n--- mensaje generado ---")
        print(detalle.messages[0].content.text)

await ver_prompts()

Fíjate en que el prompt **no responde nada**: genera el mensaje que se le va a dar al modelo.
Es andamiaje de conversación, no lógica de negocio.

## 3. Interacción: cuando el servidor necesita algo del usuario

Hasta ahora todas las llamadas se resolvían de una pasada: pregunta, respuesta. Pero a veces
el servidor **descubre a mitad de camino** que le falta un dato que solo tiene la persona.

Caso real del curso: el usuario pide una consulta que escanearía 8 GB. El servidor no debería
decidir solo si eso se paga o no.

### Cómo funciona

En vez de bloquearse esperando, el servidor **devuelve** un resultado especial:

- `resultType: "input_required"`, con la lista de lo que necesita en `inputRequests`.
- El cliente recoge la respuesta del usuario y **reintenta la petición original**, ahora con
  `inputResponses`.
- El servidor conserva su contexto entre vueltas en `requestState`.

Es un patrón de reintento, no de bloqueo. Eso es lo que permite que el protocolo sea stateless:
ninguna de las dos partes necesita mantener una conexión viva mientras el usuario decide.

Del lado del servidor, el SDK lo esconde detrás de una llamada que se lee como si fuera
síncrona (está en `curso_mcp/server.py`):

```python
estimacion = bq.estimar(sql)          # dry run: cuánto escanearía

if not estimacion.dentro_del_limite:
    respuesta = await ctx.elicit(
        message=f"Esta consulta escanearía {estimacion.megabytes:,.1f} MB. ¿La ejecuto igualmente?",
        schema=ConfirmacionDeGasto,
    )
    if respuesta.action != "accept" or not respuesta.data.continuar:
        raise RuntimeError("Consulta cancelada por el usuario.")

return bq.consultar(sql)
```

Tres detalles que importan:

1. **El dry run va primero.** Saber el coste antes de pagarlo es la razón de ser del módulo.
2. **`schema` es un modelo de Pydantic**, y el host lo convierte en un formulario nativo. No
   escribes UI.
3. **Hay tres desenlaces, no dos:** aceptar, declinar y cancelar. Tratarlos igual es un bug
   esperando su turno.

**Un cliente que contesta a lo que el servidor pregunta.** La función hace de host: recibe la
pregunta y devuelve una respuesta.

Aquí acepta siempre, para que la celda no se quede esperando. **En Claude Desktop esto sería un
formulario que ve una persona**, construido a partir del esquema de Pydantic que declara el
servidor.

**Vas a ver** primero la pregunta sobre el coste de la consulta y, tras aceptar, el resultado.
Si la consulta no supera el límite, no preguntará nada.

In [ ]:
# Un cliente que responde automáticamente a lo que pida el servidor.
# En un host real, esto es un formulario que ve una persona.
from mcp.client.session import ClientSession

async def responder_elicitacion(context, params):
    print(f"[el servidor pregunta] {params.message}")
    return {"action": "accept", "content": {"continuar": True}}

async def consulta_cara():
    sql = "SELECT COUNT(*) AS viajes FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`"
    async with Client(MI_URL, elicitation_callback=responder_elicitacion) as c:
        r = await c.call_tool("consultar", {"sql": sql})
        print(r.structured_content)

await consulta_cara()

> **Ejercicio (5 min).** Cambia el callback para que devuelva `{"action": "decline"}`.
> Comprueba que el tool devuelve un error legible y que **no** se ha ejecutado ninguna
> consulta contra BigQuery. Ese es el comportamiento correcto: cancelar significa no gastar.

## 4. Progreso y suscripciones

Una consulta que tarda cuarenta segundos sin decir nada es indistinguible de una colgada.

### Progreso

`notifications/progress` viaja **por la respuesta de la propia petición**. No hay canal
aparte: van asociadas a la llamada que las provoca.

```python
async def recorrer_tablas(ctx: Context) -> dict[str, int]:
    tablas = bq.listar_tablas()
    for i, tabla in enumerate(tablas, start=1):
        await ctx.report_progress(progress=i, total=len(tablas), message=f"Leyendo {tabla}")
        ...
```

**Progreso en directo.** `con_progreso` se ejecuta cada vez que el servidor informa de un
avance, y se la pasamos a `call_tool` como `progress_callback`.

**Vas a ver** una línea por cada tabla que el servidor va recorriendo, y al final el resultado.
Esas notificaciones llegan por la respuesta de esta misma petición, no por un canal aparte.

In [ ]:
async def con_progreso(progress, total, message):
    barra = "" if total is None else f" [{progress:.0f}/{total:.0f}]"
    print(f"…{barra} {message or ''}")

async def recorrer():
    async with Client(MI_URL) as c:
        r = await c.call_tool("recorrer_tablas", {}, progress_callback=con_progreso)
        print("\nResultado:", r.structured_content)

await recorrer()

### Suscripciones

Para lo que **no** es de una petición concreta —que cambie la lista de tools, que se actualice
un resource— existe `subscriptions/listen`: un único stream al que el cliente se suscribe
declarando qué tipos de cambio le interesan.

```python
async with client.listen(tools_list_changed=True, resources_list_changed=True) as suscripcion:
    async for evento in suscripcion:
        print("cambio:", evento)
```

### Lo que Cloud Run tiene que decir sobre esto

Un stream de larga duración choca con una realidad del despliegue:

| Límite | Valor |
|---|---|
| Timeout por defecto | **5 minutos** |
| Máximo configurable | **60 minutos** |
| Por encima de 15 min | Google recomienda asumir que el cliente se reconecta |

Y hay algo más de la revisión `2026-07-28` que conviene saber: **se eliminó la reanudación de
streams**. Si una respuesta se corta, la petición en vuelo se pierde y el cliente tiene que
reemitirla con un identificador nuevo. No hay recuperación por número de evento.

Consecuencia de diseño: **las operaciones largas no se modelan como una llamada larga.** Se
lanza el trabajo, se devuelve un identificador, y se consulta el estado. Que es exactamente
para lo que existe la extensión Tasks — la construiremos en el material de ampliación.

## Ejercicios

1. **Un resource nuevo.** Añade `catalogo://muestra/{tabla}` que devuelva las 5 primeras filas
   en Markdown. Decide con criterio su `ttlMs`: ¿cuánto puede envejecer una muestra?
2. **Baja el límite.** Pon `CURSO_MCP_LIMITE_BYTES` a `1000000` al desplegar y observa cómo
   casi cualquier consulta pide confirmación.
3. **Los tres desenlaces.** Escribe un callback que decline, otro que cancele, y comprueba que
   tu tool los distingue.

## En la próxima sesión

**Auth.** Ahora mismo tu servidor está abierto a internet y cualquiera te factura consultas.
Toca cerrarlo, y ver cómo se traduce la identidad de alguien de fuera en permisos reales
sobre BigQuery.